In [ ]:
# Import architecture from Module.py

import sys
import os
import importlib

module_dir = os.getcwd()

if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

import Module
importlib.reload(Module)

from Module import MedMultiwaveletDenoisingGenerator

print("Module path:")
print(Module.__file__)

print(MedMultiwaveletDenoisingGenerator)

In [ ]:
# Parameter Count for E1-E8

import torch




ablation_configs = {

    "E1": {
        "wavelet_type": "haar",
        "use_alignment": False,
        "use_band_refinement": False,
        "use_full_res_refinement": False,
    },

    "E2": {
        "wavelet_type": "ghm",
        "use_alignment": False,
        "use_band_refinement": False,
        "use_full_res_refinement": False,
    },

    "E3": {
        "wavelet_type": "ghm",
        "use_alignment": False,
        "use_band_refinement": True,
        "use_full_res_refinement": False,
    },

    "E4": {
        "wavelet_type": "ghm",
        "use_alignment": False,
        "use_band_refinement": False,
        "use_full_res_refinement": True,
    },

    "E5": {
        "wavelet_type": "ghm",
        "use_alignment": False,
        "use_band_refinement": True,
        "use_full_res_refinement": True,
    },

    "E6": {
        "wavelet_type": "ghm",
        "use_alignment": True,
        "use_band_refinement": False,
        "use_full_res_refinement": False,
    },

    "E7": {
        "wavelet_type": "ghm",
        "use_alignment": True,
        "use_band_refinement": True,
        "use_full_res_refinement": True,
    },

    "E8": {
        "wavelet_type": "ghm",
        "use_alignment": True,
        "use_band_refinement": True,
        "use_full_res_refinement": True,
    },
}


# Model builder

def build_ablation_model(cfg):

    return MedMultiwaveletDenoisingGenerator(
        in_ch=1,
        wavelet_type=cfg["wavelet_type"],
        band_feature_channels=40,
        feature_channels=144,
        mamba_blocks=6,
        use_alignment=cfg["use_alignment"],
        use_band_refinement=cfg["use_band_refinement"],
        use_full_res_refinement=cfg["use_full_res_refinement"],
    )


# Count parameters

print("=" * 48)
print(f"{'Exp':<8}{'Parameters':>18}{'Parameters (M)':>18}")
print("=" * 48)

parameter_results = {}

for exp, cfg in ablation_configs.items():

    model = build_ablation_model(cfg)

    params = sum(
        p.numel()
        for p in model.parameters()
    )

    params_M = params / 1e6

    parameter_results[exp] = params_M

    print(
        f"{exp:<8}"
        f"{params:>18,d}"
        f"{params_M:>18.3f}"
    )

    del model


print("=" * 48)

In [ ]:
# Diagnose E7 architecture mismatch

import inspect

cfg = ablation_configs["E7"]

print("=" * 80)
print("E7 CONFIG")
print("=" * 80)

for key, value in cfg.items():
    print(f"{key}: {value}")


print("\n" + "=" * 80)
print("GENERATOR CONSTRUCTOR")
print("=" * 80)

print(
    inspect.signature(
        MedMultiwaveletDenoisingGenerator.__init__
    )
)


print("\n" + "=" * 80)
print("BUILD FUNCTION")
print("=" * 80)

print(
    inspect.getsource(
        build_ablation_model
    )
)


print("\n" + "=" * 80)
print("PARAMETERS BY TOP-LEVEL MODULE — E7")
print("=" * 80)

model = build_ablation_model(
    ablation_configs["E7"]
)

# Exact total parameter count

exact_total = sum(
    p.numel()
    for p in model.parameters()
)

children_total = sum(
    sum(p.numel() for p in module.parameters())
    for module in model.children()
)

top_level_parameters = (
    exact_total - children_total
)

print("=" * 80)
print("PARAMETER VERIFICATION")
print("=" * 80)

print(
    f"Parameters inside child modules : "
    f"{children_total:,}"
)

print(
    f"Top-level parameters            : "
    f"{top_level_parameters:,}"
)

print(
    f"Exact model parameters          : "
    f"{exact_total:,}"
)

print(
    f"Exact model parameters (M)      : "
    f"{exact_total / 1e6:.6f} M"
)

EXPECTED_PARAMETERS = 3_682_125

if exact_total != EXPECTED_PARAMETERS:
    raise RuntimeError(
        f"E7 parameter mismatch. "
        f"Expected {EXPECTED_PARAMETERS:,}, "
        f"received {exact_total:,}."
    )

print("\nE7 parameter count: VERIFIED")

In [ ]:
# E7 RAW FLOP DIAGNOSTIC — PyTorch FlopCounterMode
# Input: 1 x 1 x 352 x 352

import torch
from torch.utils.flop_counter import FlopCounterMode

# 1. Device

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("E7 RAW FLOP DIAGNOSTIC")
print("=" * 80)
print("Device:", device)


# 2. Build the exact E7 architecture

model = build_ablation_model(
    ablation_configs["E7"]
).to(device)

model.eval()


# 3. Verify exact parameter count

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

EXPECTED_PARAMETERS = 3_682_125

print("\nParameters:")
print(f"{parameter_count:,}")
print(f"{parameter_count / 1e6:.6f} M")

if parameter_count != EXPECTED_PARAMETERS:
    raise RuntimeError(
        "E7 parameter mismatch. "
        f"Expected {EXPECTED_PARAMETERS:,}, "
        f"received {parameter_count:,}."
    )

print("Parameter count: VERIFIED")



torch.manual_seed(123)

x = torch.randn(
    1,
    1,
    352,
    352,
    device=device,
    dtype=torch.float32
)

print("\nInput:")
print("Shape:", tuple(x.shape))
print("Dtype:", x.dtype)


# Ordinary forward sanity check

with torch.inference_mode():

    y_test = model(x)

print("\nForward check:")
print("Output shape:", tuple(y_test.shape))
print(
    "Finite:",
    bool(torch.isfinite(y_test).all().item())
)

if tuple(y_test.shape) != tuple(x.shape):
    raise RuntimeError(
        "Unexpected E7 output shape. "
        f"Input={tuple(x.shape)}, "
        f"Output={tuple(y_test.shape)}."
    )

if not torch.isfinite(y_test).all():
    raise RuntimeError(
        "Non-finite values detected in E7 output."
    )


# PyTorch FlopCounterMode

print("\n" + "=" * 80)
print("FlopCounterMode report")
print("=" * 80)

with torch.inference_mode():

    with FlopCounterMode(
        display=True
    ) as flop_counter:

        y = model(x)

raw_flops = int(
    flop_counter.get_total_flops()
)

raw_gflops = (
    raw_flops / 1e9
)


#  Compare with the existing Table-4 Proposed value

TABLE_PROPOSED_FLOPS = (
    2_503_123_476_480
)

TABLE_PROPOSED_GFLOPS = (
    TABLE_PROPOSED_FLOPS / 1e9
)

difference_flops = (
    TABLE_PROPOSED_FLOPS
    - raw_flops
)

difference_gflops = (
    difference_flops / 1e9
)

ratio = (
    TABLE_PROPOSED_FLOPS
    / raw_flops
    if raw_flops > 0
    else float("nan")
)


#  Final diagnostic summary

print("\n" + "=" * 80)
print("E7 FLOP DIAGNOSTIC SUMMARY")
print("=" * 80)

print(
    f"Raw FlopCounterMode FLOPs : "
    f"{raw_flops:,}"
)

print(
    f"Raw FlopCounterMode GFLOPs: "
    f"{raw_gflops:.6f}"
)

print(
    f"Table-4 Proposed FLOPs    : "
    f"{TABLE_PROPOSED_FLOPS:,}"
)

print(
    f"Table-4 Proposed GFLOPs   : "
    f"{TABLE_PROPOSED_GFLOPS:.6f}"
)

print(
    f"Difference                : "
    f"{difference_flops:,} FLOPs"
)

print(
    f"Difference (GFLOPs)       : "
    f"{difference_gflops:.6f}"
)

print(
    f"TABLE / Raw ratio        : "
    f"{ratio:.6f}"
)

In [ ]:
# FINAL UNIFIED FLOP ANALYSIS FOR E1–E8

# FLOP convention:
#   1 multiply-add = 2 FLOPs
# Input:
#   1 x 1 x 352 x 352, FP32

import gc
import torch
import pandas as pd

from torch.utils.flop_counter import FlopCounterMode


# Global settings

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

INPUT_H = 352
INPUT_W = 352

EXPECTED_PARAMS = {
    "E1": 1_477_612,
    "E2": 2_104_504,
    "E3": 2_445_929,
    "E4": 2_137_146,
    "E5": 2_478_571,
    "E6": 3_308_058,
    "E7": 3_682_125,
    "E8": 3_682_125,
}


# 2. Fixed input

torch.manual_seed(123)

x = torch.randn(
    1,
    1,
    INPUT_H,
    INPUT_W,
    device=device,
    dtype=torch.float32,
)


# Result container

results = []


#  E1–E8

for exp_name, cfg in ablation_configs.items():

    print("\n" + "=" * 80)
    print(f"Processing {exp_name}")
    print("=" * 80)

# Build model

    model = build_ablation_model(
        cfg
    ).to(device)

    model.eval()


    for block in model.mamba_processor.blocks:
        block.mamba.use_fast_path = True


    # Parameter count

    params = sum(
        p.numel()
        for p in model.parameters()
    )

    expected = EXPECTED_PARAMS[exp_name]

    if params != expected:
        raise RuntimeError(
            f"{exp_name}: parameter mismatch. "
            f"Expected {expected:,}, "
            f"received {params:,}."
        )

    print(
        f"Parameters: {params:,} "
        f"({params / 1e6:.6f} M)"
    )


    # Ordinary 

    with torch.inference_mode():
        y_test = model(x)

    if y_test.shape != x.shape:
        raise RuntimeError(
            f"{exp_name}: unexpected output shape "
            f"{tuple(y_test.shape)}"
        )

    if not torch.isfinite(y_test).all():
        raise RuntimeError(
            f"{exp_name}: non-finite output detected."
        )


    #  PyTorch FLOPs

    with torch.inference_mode():

        with FlopCounterMode(
            display=False
        ) as flop_counter:

            _ = model(x)

    raw_flops = int(
        flop_counter.get_total_flops()
    )



    num_blocks = len(
        model.mamba_processor.blocks
    )

    mamba = (
        model.mamba_processor
        .blocks[0]
        .mamba
    )

    d_inner = int(
        mamba.d_inner
    )

    d_state = int(
        mamba.d_state
    )

    num_bands = int(
        model.expected_num_bands
    )

    # Both Haar and GHM produce H/2 x W/2
    band_h = INPUT_H // 2
    band_w = INPUT_W // 2

    sequence_length = (
        band_h * band_w
    )


    # Selective-Scan FLOPs
    #
    # B here = effective number of independent wavelet bands
    # L      = spatial sequence length
    # D      = Mamba d_inner
    # N      = d_state
    # K      = number of Mamba blocks

    B = num_bands
    L = sequence_length
    D = d_inner
    N = d_state
    K = num_blocks

    scan_mac_per_block = (
        9 * B * L * D * N
        + B * L * D       # D term
        + B * L * D       # z term
    )

    scan_mac_total = (
        K * scan_mac_per_block
    )

    scan_flops = (
        2 * scan_mac_total
    )


    # FLOPs

    final_flops = (
        raw_flops
        + scan_flops
    )


    # Print current experiment

    print(
        f"Bands                  : {B}"
    )

    print(
        f"Sequence length        : {L}"
    )

    print(
        f"Mamba blocks           : {K}"
    )

    print(
        f"d_inner                : {D}"
    )

    print(
        f"d_state                : {N}"
    )

    print(
        f"Raw FlopCounterMode    : "
        f"{raw_flops / 1e9:.6f} GFLOPs"
    )

    print(
        f"Selective-Scan addition: "
        f"{scan_flops / 1e9:.6f} GFLOPs"
    )

    print(
        f"FINAL                  : "
        f"{final_flops / 1e9:.6f} GFLOPs"
    )


# Store

    results.append({
        "Experiment": exp_name,

        "Parameters":
            params,

        "Parameters_M":
            params / 1e6,

        "Bands":
            B,

        "Sequence_Length":
            L,

        "Mamba_Blocks":
            K,

        "Raw_FlopCounterMode_FLOPs":
            raw_flops,

        "Raw_GFLOPs":
            raw_flops / 1e9,

        "Selective_Scan_FLOPs":
            scan_flops,

        "Selective_Scan_GFLOPs":
            scan_flops / 1e9,

        "Final_FLOPs":
            final_flops,

        "Final_GFLOPs":
            final_flops / 1e9,
    })


# Cleanup

    del model
    del y_test

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


#  Final table

df_flops = pd.DataFrame(
    results
)

print("\n")
print("=" * 100)
print("FINAL UNIFIED FLOP RESULTS — E1–E8")
print("=" * 100)

print(
    df_flops[
        [
            "Experiment",
            "Parameters_M",
            "Raw_GFLOPs",
            "Selective_Scan_GFLOPs",
            "Final_GFLOPs",
        ]
    ].to_string(
        index=False
    )
)


# E7 / E8 architecture consistency check

e7 = df_flops[
    df_flops["Experiment"] == "E7"
].iloc[0]

e8 = df_flops[
    df_flops["Experiment"] == "E8"
].iloc[0]

if int(e7["Parameters"]) != int(e8["Parameters"]):
    raise RuntimeError(
        "E7/E8 parameter counts differ."
    )

if int(e7["Final_FLOPs"]) != int(e8["Final_FLOPs"]):
    raise RuntimeError(
        "E7/E8 FLOPs differ."
    )




# Save

output_csv = (
    "ablation_E1_E8_flops.csv"
)

df_flops.to_csv(
    output_csv,
    index=False
)

print(
    f"\nSaved: {output_csv}"
)

In [ ]:
# LATENCY + PEAK GPU MEMORY
# E1–E8
#
# Protocol:
#   Input          : 1 x 1 x 352 x 352
#   Precision      : FP32
#   Batch size     : 1
#   Warm-up        : 20 runs
#   Measured runs  : 100 runs / trial
#   Timing trials  : 3
#   Timing         : CUDA Events
#   Memory         : torch.cuda.max_memory_allocated()
#


import gc
import time
import numpy as np
import pandas as pd
import torch

# Configuration

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for this benchmark."
    )

device = torch.device("cuda")

INPUT_H = 352
INPUT_W = 352

WARMUP_RUNS = 20
MEASURED_RUNS = 100
TIMING_TRIALS = 3

EXPECTED_PARAMS = {
    "E1": 1_477_612,
    "E2": 2_104_504,
    "E3": 2_445_929,
    "E4": 2_137_146,
    "E5": 2_478_571,
    "E6": 3_308_058,
    "E7": 3_682_125,
    "E8": 3_682_125,
}


# Fixed FP32 input

torch.manual_seed(123)

x = torch.randn(
    1,
    1,
    INPUT_H,
    INPUT_W,
    device=device,
    dtype=torch.float32,
)


#  Benchmark function

def benchmark_ablation_model(
    model,
    x,
    warmup_runs=20,
    measured_runs=100,
    timing_trials=3,
):

    model.eval()

    # Ensure normal/default Mamba execution path
    for block in model.mamba_processor.blocks:
        block.mamba.use_fast_path = True



    with torch.inference_mode():
        y = model(x)

    if y.shape != x.shape:
        raise RuntimeError(
            f"Unexpected output shape: {tuple(y.shape)}"
        )

    if not torch.isfinite(y).all():
        raise RuntimeError(
            "Non-finite output detected."
        )

    del y

    torch.cuda.synchronize()


    # Timing trials

    trial_means = []
    trial_medians = []

    for trial in range(timing_trials):

        # Warm-up
        with torch.inference_mode():

            for _ in range(warmup_runs):
                _ = model(x)

        torch.cuda.synchronize()

        times_ms = []

        # Measured passes
        with torch.inference_mode():

            for _ in range(measured_runs):

                start_event = torch.cuda.Event(
                    enable_timing=True
                )

                end_event = torch.cuda.Event(
                    enable_timing=True
                )

                start_event.record()

                _ = model(x)

                end_event.record()

                torch.cuda.synchronize()

                elapsed_ms = (
                    start_event.elapsed_time(
                        end_event
                    )
                )

                times_ms.append(
                    elapsed_ms
                )

        times_ms = np.asarray(
            times_ms,
            dtype=np.float64
        )

        trial_mean = float(
            np.mean(times_ms)
        )

        trial_median = float(
            np.median(times_ms)
        )

        trial_means.append(
            trial_mean
        )

        trial_medians.append(
            trial_median
        )

        print(
            f"  Trial {trial + 1}: "
            f"mean={trial_mean:.4f} ms, "
            f"median={trial_median:.4f} ms"
        )


    # Aggregate timing exactly across trial summaries

    trial_means = np.asarray(
        trial_means,
        dtype=np.float64
    )

    trial_medians = np.asarray(
        trial_medians,
        dtype=np.float64
    )

    inference_mean_ms = float(
        np.mean(trial_means)
    )

    inference_std_ms = float(
        np.std(
            trial_means,
            ddof=1
        )
    )

    inference_median_ms = float(
        np.median(trial_medians)
    )


    # Peak GPU memory
    # Measure inference allocation after clearing the peak

    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats(
        device
    )

    baseline_allocated = (
        torch.cuda.memory_allocated(
            device
        )
    )

    with torch.inference_mode():
        _ = model(x)

    torch.cuda.synchronize()

    peak_allocated = (
        torch.cuda.max_memory_allocated(
            device
        )
    )

    peak_total_mb = (
        peak_allocated
        / (1024 ** 2)
    )

    baseline_mb = (
        baseline_allocated
        / (1024 ** 2)
    )

    incremental_peak_mb = (
        peak_total_mb
        - baseline_mb
    )


    return {
        "inference_time_mean_ms":
            inference_mean_ms,

        "inference_time_std_ms":
            inference_std_ms,

        "inference_time_median_ms":
            inference_median_ms,

        "peak_gpu_memory_mb":
            peak_total_mb,

        "baseline_gpu_memory_mb":
            baseline_mb,

        "incremental_peak_memory_mb":
            incremental_peak_mb,
    }


# Run E1–E8

benchmark_results = []

for exp_name, cfg in ablation_configs.items():

    print("\n" + "=" * 80)
    print(f"BENCHMARKING {exp_name}")
    print("=" * 80)

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    model = build_ablation_model(
        cfg
    ).to(
        device=device,
        dtype=torch.float32
    )

    model.eval()


    # Parameter verification

    params = sum(
        p.numel()
        for p in model.parameters()
    )

    expected = EXPECTED_PARAMS[
        exp_name
    ]

    if params != expected:

        raise RuntimeError(
            f"{exp_name}: parameter mismatch. "
            f"Expected {expected:,}, "
            f"received {params:,}."
        )

    print(
        f"Parameters: "
        f"{params:,} "
        f"({params / 1e6:.6f} M)"
    )


    # Benchmark

    result = benchmark_ablation_model(
        model=model,
        x=x,
        warmup_runs=WARMUP_RUNS,
        measured_runs=MEASURED_RUNS,
        timing_trials=TIMING_TRIALS,
    )


    benchmark_results.append({

        "Experiment":
            exp_name,

        "Parameters":
            params,

        "Parameters_M":
            params / 1e6,

        "inference_time_mean_ms":
            result[
                "inference_time_mean_ms"
            ],

        "inference_time_std_ms":
            result[
                "inference_time_std_ms"
            ],

        "inference_time_median_ms":
            result[
                "inference_time_median_ms"
            ],

        "peak_gpu_memory_mb":
            result[
                "peak_gpu_memory_mb"
            ],

        "baseline_gpu_memory_mb":
            result[
                "baseline_gpu_memory_mb"
            ],

        "incremental_peak_memory_mb":
            result[
                "incremental_peak_memory_mb"
            ],

        "input_size":
            "1x1x352x352",

        "precision":
            "FP32",

        "warmup_runs":
            WARMUP_RUNS,

        "measured_runs":
            MEASURED_RUNS,

        "timing_trials":
            TIMING_TRIALS,

        "total_measured_passes":
            MEASURED_RUNS
            * TIMING_TRIALS,
    })


    print(
        f"\n{exp_name} result:"
    )

    print(
        f"Mean latency : "
        f"{result['inference_time_mean_ms']:.4f} "
        f"± "
        f"{result['inference_time_std_ms']:.4f} ms"
    )

    print(
        f"Median       : "
        f"{result['inference_time_median_ms']:.4f} ms"
    )

    print(
        f"Peak GPU mem : "
        f"{result['peak_gpu_memory_mb']:.2f} MB"
    )

    print(
        f"Baseline mem : "
        f"{result['baseline_gpu_memory_mb']:.2f} MB"
    )

    print(
        f"Incremental  : "
        f"{result['incremental_peak_memory_mb']:.2f} MB"
    )


   
    del model

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


#  Results dataframe

df_benchmark = pd.DataFrame(
    benchmark_results
)


# Merge with FLOP results

df_complexity = pd.merge(
    df_flops,
    df_benchmark[
        [
            "Experiment",
            "inference_time_mean_ms",
            "inference_time_std_ms",
            "inference_time_median_ms",
            "peak_gpu_memory_mb",
            "baseline_gpu_memory_mb",
            "incremental_peak_memory_mb",
            "input_size",
            "precision",
            "warmup_runs",
            "measured_runs",
            "timing_trials",
            "total_measured_passes",
        ]
    ],
    on="Experiment",
    how="inner",
)


# Final compact table

print("\n")
print("=" * 120)
print("FINAL UNIFIED COMPLEXITY RESULTS — E1–E8")
print("=" * 120)

print(
    df_complexity[
        [
            "Experiment",
            "Parameters_M",
            "Final_GFLOPs",
            "inference_time_mean_ms",
            "inference_time_std_ms",
            "inference_time_median_ms",
            "peak_gpu_memory_mb",
        ]
    ].to_string(
        index=False
    )
)





# Save

output_csv = (
    "ablation_E1_E8_unified_complexity.csv"
)

df_complexity.to_csv(
    output_csv,
    index=False
)

print(
    f"\nSaved: {output_csv}"
)

In [ ]:
# Inference Latency for E1-E8


import os
import gc
import cv2
import numpy as np
import torch


# Device

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for this latency protocol."
    )

device = torch.device("cuda")

print("=" * 70)
print("INFERENCE LATENCY PROTOCOL")
print("=" * 70)
print("GPU       :", torch.cuda.get_device_name(0))
print("PyTorch   :", torch.__version__)
print("Precision : FP32")
print("Batch     : 1")
print("Input     : 1 x 1 x 352 x 352")
print("Warm-up   : 30")
print("Timed runs: 100")
print("Statistic : Median")
print("=" * 70)


# Base directory

BASE = (
    "/home/sudad/HiFi_project/MultiWavelet/"
    "0-Final Work on Denoising/"
    "new architecture-mamba only"
)


# Ablation configurations

ablation_configs = {

    "E1": {
        "wavelet_type": "haar",
        "use_alignment": False,
        "use_band_refinement": False,
        "use_full_res_refinement": False,
    },

    "E2": {
        "wavelet_type": "ghm",
        "use_alignment": False,
        "use_band_refinement": False,
        "use_full_res_refinement": False,
    },

    "E3": {
        "wavelet_type": "ghm",
        "use_alignment": False,
        "use_band_refinement": True,
        "use_full_res_refinement": False,
    },

    "E4": {
        "wavelet_type": "ghm",
        "use_alignment": False,
        "use_band_refinement": False,
        "use_full_res_refinement": True,
    },

    "E5": {
        "wavelet_type": "ghm",
        "use_alignment": False,
        "use_band_refinement": True,
        "use_full_res_refinement": True,
    },

    "E6": {
        "wavelet_type": "ghm",
        "use_alignment": True,
        "use_band_refinement": False,
        "use_full_res_refinement": False,
    },

    "E7": {
        "wavelet_type": "ghm",
        "use_alignment": True,
        "use_band_refinement": True,
        "use_full_res_refinement": True,
    },

    "E8": {
        "wavelet_type": "ghm",
        "use_alignment": True,
        "use_band_refinement": True,
        "use_full_res_refinement": True,
    },
}


# Model builder

def build_ablation_model(cfg):

    model = MedMultiwaveletDenoisingGenerator(
        in_ch=1,
        out_ch=1,

        wavelet_type=cfg["wavelet_type"],

        band_feature_channels=40,
        feature_channels=144,

        mamba_blocks=6,



        use_alignment=cfg["use_alignment"],
        use_band_refinement=cfg["use_band_refinement"],
        use_full_res_refinement=cfg["use_full_res_refinement"],
    )

    return model


# Checkpoint paths

checkpoint_paths = {

    exp: os.path.join(
        BASE,
        exp,
        "exper-fusing-Frequency-baraches",
        "experiment",
        "best_ema_generator.pt"
    )

    for exp in ablation_configs
}

# Fixed noisy CT input


INPUT_IMAGE = os.path.join(
    BASE,
    "E1",
    "exper-fusing-Frequency-baraches",
    "experiment",
    "DENOISING-TEST-Results-sigma25",
    "noisy",
    "CT-0__patient102__19.png"
)


if not os.path.isfile(INPUT_IMAGE):
    raise FileNotFoundError(
        f"Input image not found:\n{INPUT_IMAGE}"
    )


img = cv2.imread(
    INPUT_IMAGE,
    cv2.IMREAD_GRAYSCALE
)


if img is None:
    raise RuntimeError(
        f"Failed to read input image:\n{INPUT_IMAGE}"
    )


if img.shape != (352, 352):
    raise ValueError(
        f"Expected image size (352,352), got {img.shape}"
    )


# uint8 -> float32 [0,1]
img = img.astype(np.float32) / 255.0


# H,W -> N,C,H,W
input_tensor = (
    torch.from_numpy(img)
    .unsqueeze(0)
    .unsqueeze(0)
    .to(
        device=device,
        dtype=torch.float32
    )
    .contiguous()
)


if tuple(input_tensor.shape) != (1, 1, 352, 352):
    raise RuntimeError(
        f"Unexpected tensor shape: {tuple(input_tensor.shape)}"
    )


if not torch.isfinite(input_tensor).all():
    raise RuntimeError(
        "Input tensor contains non-finite values."
    )


print("\nFixed input loaded successfully:")
print("Image :", os.path.basename(INPUT_IMAGE))
print("Shape :", tuple(input_tensor.shape))
print("dtype :", input_tensor.dtype)
print(
    "Range :",
    f"{input_tensor.min().item():.6f}",
    "to",
    f"{input_tensor.max().item():.6f}"
)


# Timing settings

WARMUP_RUNS = 30
TIMED_RUNS = 100


# Latency measurement

def measure_latency(
    exp_name,
    cfg,
    checkpoint_path
):

    # Verify checkpoint

    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(
            f"{exp_name} checkpoint not found:\n"
            f"{checkpoint_path}"
        )


    # Build model

    model = build_ablation_model(cfg)


    # Load EMA generator

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )


    if "gen" not in checkpoint:
        raise KeyError(
            f"{exp_name}: checkpoint has no 'gen' key."
        )


    model.load_state_dict(
        checkpoint["gen"],
        strict=True
    )


    model = model.to(device)
    model.eval()


    

    with torch.inference_mode():

        for _ in range(WARMUP_RUNS):
            output = model(input_tensor)


    torch.cuda.synchronize()


    if not torch.isfinite(output).all():
        raise RuntimeError(
            f"{exp_name}: non-finite model output."
        )


    

    starter = torch.cuda.Event(
        enable_timing=True
    )

    ender = torch.cuda.Event(
        enable_timing=True
    )


    times_ms = np.empty(
        TIMED_RUNS,
        dtype=np.float64
    )


    # Timed forward passes
    # 

    with torch.inference_mode():

        for i in range(TIMED_RUNS):

            starter.record()

            output = model(input_tensor)

            ender.record()

            # Ensure this forward pass has completed
            ender.synchronize()

            times_ms[i] = (
                starter.elapsed_time(ender)
            )


    torch.cuda.synchronize()


    # Statistics

    result = {

        "median":
            float(np.median(times_ms)),

        "mean":
            float(np.mean(times_ms)),

        "std":
            float(np.std(times_ms, ddof=1)),

        "q25":
            float(np.percentile(times_ms, 25)),

        "q75":
            float(np.percentile(times_ms, 75)),

        "min":
            float(np.min(times_ms)),

        "max":
            float(np.max(times_ms)),
    }


    # Cleanup

    del model
    del checkpoint
    del output
    del starter
    del ender

    gc.collect()

    torch.cuda.empty_cache()
    torch.cuda.synchronize()


    return result


# Measure E1-E8

latency_results = {}


print("\n")
print("=" * 88)

print(
    f"{'Exp':<6}"
    f"{'Median (ms)':>14}"
    f"{'Mean (ms)':>14}"
    f"{'Std':>12}"
    f"{'Q25':>12}"
    f"{'Q75':>12}"
)

print("=" * 88)


for exp, cfg in ablation_configs.items():

    result = measure_latency(
        exp_name=exp,
        cfg=cfg,
        checkpoint_path=checkpoint_paths[exp]
    )

    latency_results[exp] = result

    print(
        f"{exp:<6}"
        f"{result['median']:>14.3f}"
        f"{result['mean']:>14.3f}"
        f"{result['std']:>12.3f}"
        f"{result['q25']:>12.3f}"
        f"{result['q75']:>12.3f}"
    )


print("=" * 88)


# Final values for manuscript table

print("\nFINAL INFERENCE TIME (MEDIAN)")
print("=" * 38)

for exp in ablation_configs:

    print(
        f"{exp:<4}: "
        f"{latency_results[exp]['median']:.3f} ms"
    )

print("=" * 38)